# 01 — Messenger basics

`Messenger` is the scarlets SDK's agent-to-agent messaging primitive: agents
register on a shared **bus** (a name every participant agrees on) and can
`Send` a direct message, `Broadcast` to everyone on the bus, or `Receive`
whatever's addressed to them. `GatherStatus` answers "who's online right now".

This notebook uses fixed agent/bus names and a cleanup cell up front, so it's
safe to re-run from the top at any time - re-running won't leave stale state
behind from a previous run.

In [ ]:
import os
from scarlets.messaging import Messenger
from scarlets.utils.ScarletUtils import redisConnect

# Point at whatever Redis this container/notebook environment was started
# with - see examples/notebooks/.env.example for how docker-compose wires
# these in.
os.environ.setdefault("REDIS_HOST", "localhost")
os.environ.setdefault("REDIS_PORT", "6379")
os.environ.setdefault("REDIS_AUTH_TOKEN", "")
os.environ.setdefault("APP_ID", "notebook_worker")

BUS = "tutorial_headagent"

## Cleanup

Fixed names mean every run of this notebook writes to the exact same Redis
keys as the last run. Clearing them first makes re-running from the top
idempotent - no leftover messages/registrations from a previous pass.

In [ ]:
def cleanup():
    # Messenger uses different Redis keys (bus:*, tail_*, head_*) and should
    # be cleared separately - no Mapper API exists for Messenger
    r = redisConnect()
    patterns = [f"{BUS}:*", "tail_*", "head_*"]
    for pattern in patterns:
        keys = list(r.scan_iter(match=pattern))
        if keys:
            r.delete(*keys)

cleanup()
print("cleaned up")

## Registering agents on a bus

Two agents, `head` and `worker`, both join the same bus. Sharing a bus name
is the only thing that connects them - there's no separate "create the bus"
step.

In [ ]:
head = Messenger(BUS, agentId="head")
worker = Messenger(BUS, agentId="worker")

## GatherStatus — who's online

Every `Messenger` instance registers itself on construction. `GatherStatus`
reads that registry back.

In [ ]:
status = head.GatherStatus()
status

## Send / Receive — direct messaging

`Send` addresses a message to one agent by its `agentId`. `Receive` blocks
(up to `timeout` seconds) until something addressed to this agent arrives.

In [ ]:
head.Send("worker", {"task": "ping", "data": "hello from head"})

msg = worker.Receive(timeout=5)
msg

## Broadcast — reaching every agent on the bus

A second worker joins, then `head` broadcasts once and both workers receive
their own copy.

In [ ]:
worker2 = Messenger(BUS, agentId="worker2")

head.Broadcast({"directive": "status_check"})

print("worker:", worker.Receive(timeout=5))
print("worker2:", worker2.Receive(timeout=5))

## AsTools — exposing Messenger to an LLM agent loop

`AsTools()` returns a tool-call schema (`tools`) plus the handler functions
(`handlers`) that implement them - built for wiring a `Messenger` straight
into an LLM's tool-calling loop without hand-writing the glue.

In [ ]:
tools = head.AsTools()
[t["name"] for t in tools["tools"]]

## Cleanup (teardown)

Run the same cleanup again so this notebook doesn't leave anything behind
for the next tutorial.

In [ ]:
cleanup()
print("cleaned up")